In [27]:
from sage.all import *

# ============================================================
# Parameters
# ============================================================

n = 5                  # 4 variables -> 4 output polynomials
p = 2                  # GF(2) for Boolean functions
S = 5                  # Number of composition iterations
T = 100000                # Maximum attempts to obtain a basis of size n


F = GF(p)

print(f"Field: GF({p})")
print(f"Number of variables: {n}")
print(f"Composition depth S: {S}")

Field: GF(2)
Number of variables: 5
Composition depth S: 5


In [28]:
# Base polynomial ring
R = PolynomialRing(
    F,
    names=[f"x{i}" for i in range(n)],
    order="degrevlex"
)

x = R.gens()

print("Variables:")
print(x)

Variables:
(x0, x1, x2, x3, x4)


In [29]:
def random_quadratic(R):
    """
    Generate a random quadratic polynomial over GF(2).

    Includes:
        constant terms
        linear terms
        quadratic terms
    """

    variables = R.gens()
    n = len(variables)

    f = R.zero()

    # Constant term
    f += R.random_element(2)

    # Linear terms
    for i in range(n):
        if ZZ.random_element(2):
            f += variables[i]

    # Quadratic terms
    for i in range(n):
        for j in range(i, n):
            if ZZ.random_element(2):
                f += variables[i] * variables[j]

    return R(f)

In [30]:
def random_quadratic_system(R, n):
    """
    Generate n random quadratic polynomials.
    """

    return [
        random_quadratic(R)
        for _ in range(n)
    ]

In [31]:
def generate_grobner_basis(R, n, T=100):
    """
    Repeatedly generate random quadratic systems until
    the reduced Groebner basis contains exactly n polynomials.
    """

    for attempt in range(1, T + 1):

        G = random_quadratic_system(R, n)

        ideal = R.ideal(G)

        B = ideal.groebner_basis()

        if len(B) == n:
            return B, G, attempt

    raise RuntimeError(
        f"Failed to obtain a Groebner basis of size {n} "
        f"after {T} attempts."
    )

In [32]:
B0, G0, attempts = generate_grobner_basis(
    R,
    n,
    T
)

F_current = list(B0)

print(f"Found basis after {attempts} attempt(s)")
print("\nInitial Groebner basis F^(0):")

for i, f in enumerate(F_current):
    print(f"f{i} =", f)

RuntimeError: Failed to obtain a Groebner basis of size 5 after 100000 attempts.

In [ ]:
def shuffle_polynomials(F_current):
    """
    Randomly shuffle the current polynomial system.
    """

    permutation = list(range(len(F_current)))
    shuffle(permutation)

    shuffled = [
        F_current[i]
        for i in permutation
    ]

    return shuffled, permutation

In [ ]:
def compose_polynomials(B, F_previous, R):
    """
    Compose each polynomial in B with the previous
    polynomial system.

    B:
        New Groebner basis in variables x0,...,x3

    F_previous:
        Previous polynomial system
    """

    x = R.gens()

    composed = []

    for b in B:

        substitutions = {
            x[i]: F_previous[i]
            for i in range(len(x))
        }

        new_poly = b.subs(substitutions)

        composed.append(R(new_poly))

    return composed

In [ ]:
def iterated_grobner_composition(
    R,
    n,
    S,
    T=100
):
    """
    Iterated Groebner Basis Composition with Shuffle.

    Adapted from the Bachelor's project algorithm.

    Returns:
        F^(S)
        history of all intermediate systems
        permutations used
    """

    # --------------------------------------------------------
    # Phase 1: Initialisation
    # --------------------------------------------------------

    B, G, attempts = generate_grobner_basis(
        R,
        n,
        T
    )

    F_current = list(B)

    history = {
        0: list(F_current)
    }

    permutations = {}

    # --------------------------------------------------------
    # Phase 2: Composition Loop
    # --------------------------------------------------------

    for k in range(1, S + 1):

        # Generate new random quadratic system
        B, G, attempts = generate_grobner_basis(
            R,
            n,
            T
        )

        # Shuffle previous polynomial system
        F_shuffled, sigma = shuffle_polynomials(
            F_current
        )

        # Compose
        F_new = compose_polynomials(
            B,
            F_shuffled,
            R
        )

        # Store
        F_current = F_new

        history[k] = list(F_current)
        permutations[k] = sigma

        print(
            f"Iteration {k}: "
            f"basis found after {attempts} attempt(s), "
            f"shuffle = {sigma}"
        )

    return F_current, history, permutations

In [ ]:
F_final, history, permutations = iterated_grobner_composition(
    R=R,
    n=n,
    S=S,
    T=T
)

print("\n" + "=" * 60)
print(f"FINAL POLYNOMIAL SYSTEM F^({S})")
print("=" * 60)

for i, f in enumerate(F_final):
    print(f"\nf{i}(x0,x1,x2,x3) =")
    print(f)

AttributeError: type object 'R' has no attribute 'gens'

In [ ]:
def polynomial_function(polynomials, bits):
    """
    Evaluate a polynomial system on a 4-bit input.

    Example:
        bits = "1011"

    Returns:
        4-bit output string.
    """

    if len(bits) != 4 or any(b not in "01" for b in bits):
        raise ValueError("Input must be exactly 4 bits.")

    values = {
        R.gen(i): F(bits[i])
        for i in range(4)
    }

    output = []

    for polynomial in polynomials:
        value = polynomial.subs(values)
        output.append(str(int(value)))

    return "".join(output)

In [ ]:
for bits in ["0000", "0001", "0010", "1010", "1111"]:

    output = polynomial_function(
        F_final,
        bits
    )

    print(
        f"{bits} -> {output}"
    )

0000 -> 0001
0001 -> 1000
0010 -> 0000
1010 -> 0111
1111 -> 0110


In [ ]:
def truth_table(polynomials):
    """
    Generate the complete 4-bit truth table.
    """

    table = []

    for value in range(16):

        bits = format(value, "04b")

        output = polynomial_function(
            polynomials,
            bits
        )

        table.append(
            (bits, output)
        )

    return table

In [ ]:
table = truth_table(F_final)

print("Input -> Output")
print("-" * 20)

for x_in, y_out in table:
    print(f"{x_in} -> {y_out}")

Input -> Output
--------------------
0000 -> 0001
0001 -> 1000
0010 -> 0000
0011 -> 1001
0100 -> 1001
0101 -> 1010
0110 -> 1000
0111 -> 1011
1000 -> 0110
1001 -> 0101
1010 -> 0111
1011 -> 0100
1100 -> 1110
1101 -> 0111
1110 -> 1111
1111 -> 0110


In [ ]:
outputs = [
    output
    for _, output in table
]

print("Number of unique outputs:", len(set(outputs)))

if len(set(outputs)) == 16:
    print("✓ The function is bijective.")
else:
    print("✗ The function is NOT bijective.")

Number of unique outputs: 12
✗ The function is NOT bijective.
